# 01 · Customer fleet ownership

This notebook replicates the Customer fleet ownership paragraph of the report's Results section.

> Customer fleet ownership: Ticket serials reach 3,041 of 7,183 devices (42%), and accounts are individual facilities (845 accounts at 725–837 sites), so fleets come from the installed-base list.

**How it works.** We count the clearView devices that ticket serials reach in the per-device table from the telemetry database, and we count the accounts and their sites in the tables that `customer_fleet_audit.py` builds from IPA's case export. With `RERUN = True`, the notebook re-runs that script in a scratch copy of the outputs and compares each file it writes with the committed one; with `RERUN = False`, it reads the committed outputs. Either way, it recomputes every number from the tables in `OUT` and checks it against the report.

**Re-run.** With `RERUN = True`, we re-run `verification/customer_fleet_audit.py`, which takes about 3 s. It rewrites the account and site tables (`fleet_*.csv` at the top of `verification/outputs/`) in a scratch copy of the outputs, and it skips the script's optional checks on the telemetry database, as we explain below the comparison. `docs/03_customer_fleet_ownership.md` (§3.1 and §3.4) describes both analyses.

**Used as committed, without re-running.** We read the telemetry-database outputs as committed, because the scripts that wrote them need the clearView telemetry database, which the notebooks do not re-read. They are `fu/fu_devices.csv` and `fu/fu_ticket_last10.csv` from `db_followup.py`, and `db/db_devices_summary.csv`, `db/db_linkage.csv`, and `db/db_record_quality.csv` from `verify_with_db.py`.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    # The committed fleet_*.csv hold no results of the script's telemetry-database checks (D1-D7), which it runs
    # whenever its --db path exists, so we give it a path that does not exist
    no_database = OUT / 'no_telemetry_database'
    assert not no_database.exists()
    R.rerun(OUT, 'customer_fleet_audit.py', '--db', str(no_database))
R.compare(OUT)

customer_fleet_audit.py --db OUT/no_telemetry_database: exit status 0, 4 s
    done
15 files written: 8 DIFFERENT, 7 identical.
    fleet_account_granularity.csv: DIFFERENT (5 vs. 11 rows; columns same)
    fleet_customer_naive_rates.csv: DIFFERENT (columns in_name_tier_multi_account_system, in_reviewed_multi_account_system)
    fleet_facility_vs_system.csv: DIFFERENT (column value)
    fleet_host_expectations.csv: DIFFERENT (6 vs. 13 rows; columns same)
    fleet_multi_account_serial_classes.csv: DIFFERENT (columns relationship, rule_class, serials)
    fleet_multi_account_serials.csv: DIFFERENT (columns relationship, rule_class, same_system_name_tier, same_system_reviewed)
    fleet_shared_addresses.csv: DIFFERENT (columns same_name_tier_system, same_reviewed_system)
    fleet_system_size_distribution.csv: DIFFERENT (10 vs. 20 rows; columns same)


,file written by the re-run,compared with the committed file
0,fleet_account_granularity.csv,DIFFERENT (5 vs. 11 rows; columns same)
1,fleet_coverage_bounds.csv,identical
2,fleet_customer_naive_rates.csv,"DIFFERENT (columns in_name_tier_multi_account_system, in_reviewed_multi_account_system)"
3,fleet_db_checks_spec.csv,identical
4,fleet_facility_vs_system.csv,DIFFERENT (column value)
5,fleet_host_expectations.csv,DIFFERENT (6 vs. 13 rows; columns same)
6,fleet_model_type_note.csv,identical
7,fleet_multi_account_serial_classes.csv,"DIFFERENT (columns relationship, rule_class, serials)"
8,fleet_multi_account_serials.csv,"DIFFERENT (columns relationship, rule_class, same_system_name_tier, same_system_reviewed)"
9,fleet_naive_rate_summary.csv,identical


**About the comparison.** The committed `fleet_*.csv` come from a run without the telemetry database, so none of them holds results of the script's database checks; `fleet_db_checks_spec.csv` holds only the checks' specification, which the script writes on every run. The script runs checks D1–D7 whenever its `--db` path exists, and that path defaults to the database's place in `data/raw/`, so we pass `--db` a path in the scratch folder that does not exist. Otherwise, on a computer that keeps the database with IPA's raw files, the re-run would read the database's device table and write nine `fleet_db_D*.csv` files that have no committed copy. The checks add only those files to the outputs, so skipping them changes none of the other tables.

Besides IPA's case export, `customer_fleet_audit.py` reads a hand-made review, kept in the private folder, that groups the accounts into health systems, and the committed files were made with it. Where that file is missing, as in the run saved here, the script falls back to its name-token grouping, and the files that carry the reviewed grouping are listed as DIFFERENT because their reviewed-system rows or columns change: `fleet_account_granularity.csv`, `fleet_system_size_distribution.csv`, `fleet_shared_addresses.csv`, `fleet_customer_naive_rates.csv`, `fleet_multi_account_serials.csv`, `fleet_multi_account_serial_classes.csv`, `fleet_facility_vs_system.csv`, and `fleet_host_expectations.csv`. Without the review, the script does not write `fleet_segments.csv`, `fleet_review_evidence.csv`, or `fleet_l2b_cluster_review.csv`, so their committed copies stay in the scratch folder. The account and site counts do not depend on the review, so the parts of these files that this notebook reads (the accounts and their tickets, the accounts at shared addresses, and the account and site rows of the granularity table) are the same in the re-created and committed files.

## Devices that ticket serials reach

A device is a row of the clearView device table, and a ticket serial reaches it when the device ID equals the Serial # on at least one ticket in IPA's case export, as exact text after trimming spaces. The count uses tickets of every kind, including the planned work, canceled requests, and duplicates that the Definitions leave out of the unplanned tickets, so we also count the devices that unplanned tickets alone reach. We count the devices in `fu/fu_devices.csv` (one row per device, from `db_followup.py`), and a device counts as reached when it has a ticket account, which the script fills only when the device ID is the serial on a ticket. We take each device-matched ticket and its kind from `fu/fu_ticket_last10.csv`, which the same run of `db_followup.py` wrote, and we assert that their device IDs are the reached devices. We also assert that the counts agree with `verify_with_db.py`'s separate run, which counted the rows of the device table and ran the midterm analysis's linkage query over every ticket (`db/db_devices_summary.csv` and `db/db_linkage.csv`).

In [3]:
devices = R.read_csv(OUT, 'fu/fu_devices.csv')
assert devices.machine_id.is_unique
n_devices = len(devices)
reached = devices.ticket_customer.notna()
n_reached = int(reached.sum())
share_reached = n_reached / n_devices

# The device-matched tickets of every kind, once each (the table repeats each ticket for every telemetry window)
tickets = R.read_csv(OUT, 'fu/fu_ticket_last10.csv', usecols=['case_id', 'machine_id', 'service_group'])
tickets = tickets.drop_duplicates()
assert tickets.case_id.is_unique  # each ticket has one device and one kind in every window
unplanned = ~tickets.service_group.isin(['Planned work', 'Cancelled request', 'Duplicate case'])
n_reached_unplanned = tickets.machine_id[unplanned].nunique()

# Cross-checks: the device IDs on those tickets (same db_followup.py run), and verify_with_db.py's separate run
assert set(tickets.machine_id) == set(devices.machine_id[reached])
device_summary = R.read_csv(OUT, 'db/db_devices_summary.csv').iloc[0]
linkage = R.read_csv(OUT, 'db/db_linkage.csv').iloc[0]
assert int(device_summary['rows']) == n_devices, 'verify_with_db.py counts a different number of device-table rows'
assert int(linkage['matched_machines']) == n_reached, 'the linkage query reaches a different number of devices'
assert int(linkage['matched_cases']) == len(tickets), 'the linkage query matches a different number of tickets'
n_tickets = int(linkage['cases'])

print(f'Ticket serials on the {n_tickets:,} tickets of every kind reach {n_reached:,} of {n_devices:,} devices '
      f'({share_reached:.1%}).')
print(f'Unplanned tickets alone reach {n_reached_unplanned:,} devices ({n_reached_unplanned / n_devices:.1%}), and '
      f'the other {n_reached - n_reached_unplanned} reached devices have only planned work, canceled requests, '
      f'or duplicates.')
pd.DataFrame([
    ('fu/fu_devices.csv', 'devices, and those with a ticket account', R.num(n_devices), R.num(n_reached)),
    ('fu/fu_ticket_last10.csv', 'device IDs on the device-matched tickets', '', R.num(tickets.machine_id.nunique())),
    ('db/db_devices_summary.csv, db/db_linkage.csv', 'device-table rows, and the linkage query',
     R.num(int(device_summary['rows'])), R.num(int(linkage['matched_machines']))),
], columns=['table', 'what it counts', 'devices', 'reached by a ticket serial'])

Ticket serials on the 7,471 tickets of every kind reach 3,041 of 7,183 devices (42.3%).
Unplanned tickets alone reach 2,993 devices (41.7%), and the other 48 reached devices have only planned work, canceled requests, or duplicates.


,table,what it counts,devices,reached by a ticket serial
0,fu/fu_devices.csv,"devices, and those with a ticket account","7,183","3,041"
1,fu/fu_ticket_last10.csv,device IDs on the device-matched tickets,,"3,041"
2,"db/db_devices_summary.csv, db/db_linkage.csv","device-table rows, and the linkage query","7,183","3,041"


## Accounts

An account is a ticket's Account Name after trimming spaces, which the midterm analysis calls a customer. We count the rows of `fleet_customer_naive_rates.csv`, which `customer_fleet_audit.py` writes with one row per account and its tickets of every kind. We assert that the count and the accounts' tickets agree with `verify_with_db.py`'s run of the midterm analysis's record-quality query (`db/db_record_quality.csv`), which counts the distinct customers and the tickets in the export.

In [4]:
accounts = R.read_csv(OUT, 'fleet_customer_naive_rates.csv')
assert accounts.customer.is_unique
n_accounts = len(accounts)

# Cross-check: the midterm analysis's record-quality query, from verify_with_db.py's separate run
record_quality = R.read_csv(OUT, 'db/db_record_quality.csv').iloc[0]
assert int(record_quality['customers']) == n_accounts, 'verify_with_db.py counts a different number of customers'
assert accounts.tickets.sum() == int(record_quality['cases']) == n_tickets, 'the ticket counts differ'
print(f'{n_accounts:,} accounts, which carry {accounts.tickets.sum():,} tickets.')

845 accounts, which carry 7,471 tickets.


## Sites: distinct address strings

Each account has one address string, which has no street line: a city and state, then a ZIP code (with or without the +4) or a Canadian postal code. `customer_fleet_audit.py` stops if an account has more than one. The distinct address strings therefore number the accounts minus the extra accounts at shared addresses, which we take from `fleet_shared_addresses.csv`. It has one row per address string that more than one account shares, with its accounts as pseudonyms and the precision of its code, which is either a ZIP+4 or postal code or only a 5-digit ZIP.

In [5]:
shared = R.read_csv(OUT, 'fleet_shared_addresses.csv')
shared['accounts at the address'] = shared.customers.str.split(' + ', regex=False).str.len()
extra_accounts = int((shared['accounts at the address'] - 1).sum())
n_addresses = n_accounts - extra_accounts
print(f'{n_accounts:,} accounts minus {extra_accounts} extra accounts at the {len(shared)} shared address strings '
      f'leaves {n_addresses:,} distinct address strings.')
shared.groupby(['accounts at the address', 'address_precision']).size().rename('shared address strings').reset_index()

845 accounts minus 8 extra accounts at the 8 shared address strings leaves 837 distinct address strings.


,accounts at the address,address_precision,shared address strings
0,2,ZIP+4/postal code,7
1,2,ZIP5 only,1


## Sites: distinct City, State, and ZIP5

The lower end of the range is the number of distinct city, state, and 5-digit ZIP combinations, a lower bound on sites because it merges different organizations that share a ZIP5. For the Canadian accounts, `customer_fleet_audit.py` uses the full postal code in place of the ZIP5, so it merges them only when they share a postal code. No committed table lists the accounts' addresses, which are private, so we read this count from `fleet_account_granularity.csv`, where `customer_fleet_audit.py` counts it over the same accounts. The same table counts the accounts and the distinct address strings, and we assert that both equal our recounts above.

In [6]:
granularity = R.read_csv(OUT, 'fleet_account_granularity.csv')


def granularity_count(rule):
    """The count in the one granularity row whose rule starts with `rule`."""
    row = granularity[granularity.rule.str.startswith(rule)]
    assert len(row) == 1, rule
    return int(row['count'].iloc[0])


n_zip5_sites = granularity_count('distinct City+State+ZIP5')
assert granularity_count('distinct cleaned Account Name') == n_accounts, 'the account counts differ'
assert granularity_count('distinct address string') == n_addresses, 'the address-string counts differ'
assert n_zip5_sites <= n_addresses  # each address string gives one City, State, and ZIP5
print(f'{n_zip5_sites:,} distinct City + State + ZIP5 combinations among the {n_accounts:,} accounts.')
pd.DataFrame({'fleet_account_granularity.csv': [granularity_count('distinct cleaned Account Name'),
                                                granularity_count('distinct address string')],
              'recount above': [n_accounts, n_addresses]},
             index=['accounts', 'distinct address strings'])

725 distinct City + State + ZIP5 combinations among the 845 accounts.


,fleet_account_granularity.csv,recount above
accounts,845,845
distinct address strings,837,837


## Check against the report

In [7]:
R.check([
    ('Devices that ticket serials reach', '3,041', R.num(n_reached)),
    ('Devices in the telemetry device table', '7,183', R.num(n_devices)),
    ('Share of devices reached', '42%', R.pct(share_reached)),
    ('Accounts', '845', R.num(n_accounts)),
    ('Sites, ZIP5 combinations to address strings', '725–837', R.span(n_zip5_sites, n_addresses)),
]);

,Finding,Report,Recomputed,Match
0,Devices that ticket serials reach,"3,041","3,041",yes
1,Devices in the telemetry device table,"7,183","7,183",yes
2,Share of devices reached,42%,42%,yes
3,Accounts,845,845,yes
4,"Sites, ZIP5 combinations to address strings",725–837,725–837,yes


All 5 values match the report.
